In [18]:
import random
import math
import ipywidgets as widgets
from IPython.display import display, clear_output
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

This cell imports all the necessary libraries for the project:

**random & math:** Used for the genetic algorithm logic and calculating distances.

**ipywidgets & IPython.display:** Used to create the interactive sliders and buttons you see in the dashboard.

**geopandas:** This is a special library used to read and process the geographical map file (India.geojson).

**pandas:** Used for data handling.

**matplotlib.pyplot:** The standard library for drawing the plots and the map of India.

In [19]:
cities_coords = {
    'Delhi': (28.6139, 77.2090),
    'Mumbai': (19.0760, 72.8777),
    'Kolkata': (22.5726, 88.3639),
    'Chennai': (13.0827, 80.2707),
    'Bengaluru': (12.9716, 77.5946),
    'Hyderabad': (17.3850, 78.4867),
    'Ahmedabad': (23.0225, 72.5714),
    'Guwahati': (26.1445, 91.7362),
    'Bhopal': (23.2599, 77.4126),
    'Chandigarh': (30.7333, 76.7794),
    'Jaipur': (26.9225, 75.7701),
    'Lucknow': (26.8467, 80.9462),
    'Patna': (25.6981, 85.1398),
    'Nagpur': (21.1458, 79.0882),
    'visakhapatnam': (17.6868, 83.2185)

}

This cell defines a dictionary called cities_coords.

It contains the Latitude and Longitude for 15 major Indian cities (like Delhi, Mumbai, and Kolkata).
These coordinates are essential because the algorithm uses them to calculate the real-world distance between cities and to accurately place the red dots on the map of India.

In [42]:
city_mapping = {
    '0000'   :'Mumbai',
    '0001'   :'Kolkata',
    '0010'   :'Chennai',
    '0100'   :'Bengaluru',
    '1000'   :'Hyderabad',
    '1001'   :'Ahmedabad',
    '1010'   :'Guwahati',
    '1100'   :'Bhopal',
    '1101'   :'Chandigarh',
    '1110'   :'Jaipur',
    '1111'   :'Lucknow',
    '0011'   :'Nagpur',
    '0101'   :'Patna',
    '0111'   :'visakhapatnam'
    # Delhi removed from here as it is our fixed start/end point
}

This cell defines the `city_mapping` dictionary, which assigns a unique 4-bit binary string to each of the 15 cities. This allows the Genetic Algorithm to represent a full travel route as a single long string of bits (a 'chromosome').

In [46]:
def haversine(c1, c2):
    coord1, coord2 = cities_coords[c1], cities_coords[c2]
    R = 6371
    lat1, lon1 = math.radians(coord1[0]), math.radians(coord1[1])
    lat2, lon2 = math.radians(coord2[0]), math.radians(coord2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

### Explaining the Haversine Function

This function calculates the distance between two points on a sphere given their longitudes and latitudes.

1.  **Coordinates:** It looks up the lat/long for two cities from the `cities_coords` dictionary.
2.  **Conversion:** It converts degrees to radians (math required for spherical geometry).
3.  **The Formula:** It uses the Haversine formula with the Earth's radius (~6,371 km) to find the shortest distance between those two points.

### Breakdown of Haversine Variables:

*   **`math.radians()`**: Converts degrees from your `cities_coords` into radians because Python's math functions (like `sin` and `cos`) require radians.
*   **`a = sin²(Δlat/2) + cos(lat1) ⋅ cos(lat2) ⋅ sin²(Δlong/2)`**: This part of the formula calculates the square of half the straight-line distance between the points.
*   **`atan2`**: This is used to find the inverse tangent, providing the central angle between the two points.
*   **`R * c`**: The final conversion from an angle into a physical distance (km) on the Earth's surface.

In [50]:
all_locs = list(set(['Delhi'] + list(city_mapping.values())))
dist_matrix = {}
for c1 in all_locs:
    for c2 in all_locs:
        if (c2, c1) in dist_matrix:
            dist_matrix[(c1, c2)] = dist_matrix[(c2, c1)]
        else:
            dist_matrix[(c1, c2)] = haversine(c1, c2)

### Understanding `all_locs`

This list is generated by combining the starting city with the target cities:
*   **`city_mapping.values()`**: Retrieves the names (Mumbai, Kolkata, etc.) from our binary mapping.
*   **`['Delhi'] + ...`**: Explicitly adds Delhi to the pool, as it is our fixed starting and ending point.
*   **`set(...)`**: A safety measure that removes any potential duplicate names before we start calculating distances.

### The Distance Matrix

This cell pre-calculates the distances between all pairs of cities and stores them in `dist_matrix`.

*   **Why?** Calculating trigonometry (Haversine) is computationally expensive. By calculating it once and storing it, we save thousands of calculations during the evolution process.
*   **Symmetry:** Notice the `if (c2, c1) in dist_matrix` check—this ensures that the distance from Delhi to Mumbai is correctly identified as the same as Mumbai to Delhi, avoiding redundant math.

In [51]:
def create_individual():
    # Now we sample 14 cities since Delhi is fixed
    return ''.join(random.sample(list(city_mapping.keys()), 14))

### Creating an Individual

In a Genetic Algorithm, an **Individual** is a single potential solution. In our case, an individual is a random sequence of cities.

*   **`random.sample(..., 15)`**: This ensures we pick all 15 cities exactly once, which prevents the traveler from visiting the same city twice (a common constraint in the Traveling Salesperson Problem).
*   **`''.join(...)`**: This turns the list of binary codes into one long string of 60 bits (15 cities × 4 bits each), which is the format the algorithm uses to evolve.

In [52]:
def create_population(size: int = 10):
    population = list()
    while len(population) < size:
        individual = create_individual()
        if individual not in population:
            population.append(individual)

    return population

### Creating the Population

This function builds a starting group of solutions.

*   **`population = list()`**: An empty list to store our random travelers.
*   **`while len(population) < size`**: Continues until we have the desired number of unique routes.
*   **`if individual not in population`**: This is a crucial check. If we randomly generated the same route twice, we ignore the duplicate. This keeps the initial population diverse, which helps the algorithm explore more of India.

In [53]:
def calculate_individual_dist(individual, city_mapping, start_city, distance_matrix:dict[tuple,float]):
    # Individual is now 14 cities * 4 bits = 56 bits
    city_keys = [individual[i:i+4] for i in range(0, 56, 4)]
    path_names = [start_city] + [city_mapping[k] for k in city_keys] + [start_city]
    total_dist = 0
    for i in range(len(path_names) -1):
        c1 = path_names[i]
        c2 = path_names[i+1]
        total_dist += distance_matrix.get((c1, c2), 0)
    return total_dist

### Measuring a Route

This function calculates the total distance of a journey represented by a 60-bit string.

*   **`path_names`**: This creates the full loop: `Delhi` -> `City 1` -> `City 2` ... -> `City 15` -> `Delhi`.
*   **`total_dist`**: It iterates through the sequence, adding up the distance between each pair of consecutive cities using our pre-calculated matrix.
*   **Fitness**: In Genetic Algorithm terms, this calculates the **cost**. The goal of the algorithm is to minimize this value.

In [54]:
def calculate_cost_for_population(population, city_mapping, start_city, dist_matrix):
    population_with_cost  = []

    for individual in population:
        cost = calculate_individual_dist(individual, city_mapping, 'Delhi', dist_matrix)
        population_with_cost.append((individual, cost))

        # population_with_cost.sort(key=lambda x: x[1])

    return population_with_cost

### Evaluating the Group

This function bridges individual measurement with population management.

*   **Looping**: It iterates through every individual in the `population` list.
*   **Pairing**: It stores the result as a tuple: `(route_string, total_km)`.
*   **Ranking**: Although sorting is commented out here (it's often handled later during selection), this data structure is what allows us to pick the 'best' route for the visualization at the end of every generation.

### Understanding `key = lambda x: x[1]`

You'll notice this line used when we calculate the population cost or run the tournament. Here is what it means:

In our algorithm, the population is stored as a list of **tuples**:
`[('101001...', 4500.5), ('001110...', 3200.2), ...]`

*   Each element `x` in the list is a pair: `(binary_string, total_distance)`.
*   `x[0]` refers to the binary string (the route).
*   `x[1]` refers to the distance (the cost).

When we use `min(population, key=lambda x: x[1])`, we are telling Python:
> "Don't compare the long strings of 1's and 0's (`x[0]`). Instead, look at the **second item** (`x[1]`) in every pair and find the smallest number."

Without this `key`, Python wouldn't know whether you want to find the 'minimum' route based on alphabetical order or based on the distance!

In [27]:
def tournament_selection(population_with_cost, tournament_size=2):
    contestent = random.sample(population_with_cost, tournament_size)
    winner = min(contestent, key= lambda x: x[1])
    return winner

### Understanding Tournament Selection

This function is used to choose which individuals (routes) get to pass their 'genes' to the next generation. It mimics natural selection through a small competition:

1.  **Sampling**: It randomly picks a small group of individuals from the population (defined by `tournament_size`).
2.  **The Contest**: It compares the costs (total distance) of these few individuals.
3.  **The Winner**: The individual with the **lowest** cost (the shortest route) wins the tournament and is returned as a parent.

**Why use this?** It provides a good balance between 'selection pressure' (favoring the best) and 'diversity' (allowing slightly weaker candidates a small chance to win if they happen to be in a weak tournament), which prevents the algorithm from getting stuck in a local minimum too early.

In [45]:
def single_point_crossover(parent1, parent2, city_mapping, crossover_rate):
    if random.random() > crossover_rate:
        return parent1, parent2

    # 14 chunks of 4 bits
    p1_chunks = [parent1[i:i+4] for i in range(0, 56, 4)]
    p2_chunks = [parent2[i:i+4] for i in range(0, 56, 4)]

    k = random.randint(1, 13)

    def cross_and_repair(parent_head, parent_tail_source):
        child = parent_head[:k] + parent_tail_source[k:]
        missing = list(set(city_mapping.keys()) - set(child))
        random.shuffle(missing)
        seen = set(parent_head[:k])
        for i in range(k, 14):
            city = child[i]
            if city in seen:
                child[i] = missing.pop()
                seen.add(child[i])
            else:
                seen.add(city)
        return "".join(child)

    c1 = cross_and_repair(p1_chunks, p2_chunks)
    c2 = cross_and_repair(p2_chunks, p1_chunks)
    return c1, c2

### Explaining the Crossover Logic (Reproduction)

In a standard Genetic Algorithm, crossover usually just swaps halves of strings. However, in our TSP problem, if we simply swap halves, we might end up with a route that visits **Mumbai twice** and **never visits Delhi**. To prevent this, we use a **Crossover with Repair** strategy:

1.  **Splitting**: We pick a random point (`k`) in the 15-city sequence.
2.  **Inheritance**: The child takes the first `k` cities directly from Parent A.
3.  **Filling**: The child then takes the remaining cities from Parent B to fill the rest of the route.
4.  **The Repair Step**:
    *   The algorithm checks if any cities from Parent B were already present in the segment taken from Parent A.
    *   If a duplicate is found, it identifies which cities are "missing" from the overall 15-city list.
    *   It replaces the duplicate city with one of the missing cities.

**Outcome**: This ensures every child is a **valid permutation** (a legal route where every city is visited exactly once) while still inheriting geographical 'traits' from both parents.

In [55]:
def mutation(individual, mutation_rate=0.02):
    if random.random() > mutation_rate:
        return individual
    # 14 chunks of 4 bits
    chunks = [individual[i:i+4] for i in range(0, 56, 4)]
    idx1, idx2 = random.sample(range(len(chunks)), 2)
    chunks[idx1], chunks[idx2] = chunks[idx2], chunks[idx1]
    return "".join(chunks)

### Maintaining Diversity with Mutation

Mutation acts as a 'random shuffle' that prevents the algorithm from getting stuck in a local optimum. In this project, we use **Swap Mutation**:

1.  **Probability Check**: The algorithm only performs a mutation if a random number is less than the `mutation_rate` (usually very low, like 1% or 2%).
2.  **Binary Decoding**: The 60-bit string is split back into its 15 city chunks (4 bits each).
3.  **The Swap**: It randomly selects two different positions (cities) in the route and swaps their places.
4.  **Re-encoding**: The modified list is joined back into a single 60-bit string.

**Why this specific method?**
Because we are solving a permutation problem (TSP), we cannot simply flip bits (which might create invalid city codes). Swapping two existing cities ensures the route remains valid (visiting every city once) while effectively testing a brand new path variation.

In [56]:
plot_output = widgets.Output()
india_map = gpd.read_file("/content/India.geojson")

def run_ga(pop_size, generations, crossover_rate, mutation_rate, tournament_size):
    fig, ax = plt.subplots(figsize=(10,12))
    if india_map is not None:
        india_map.plot(ax=ax, color='white', edgecolor='black', linewidth=0.5)

    lons = [v[1] for v in cities_coords.values()]
    lats = [v[0] for v in cities_coords.values()]
    ax.scatter(lons, lats, color='red', s=50, zorder=5, label='Cities')

    for city, coord in cities_coords.items():
        ax.text(coord[1]+0.3, coord[0], city, fontsize=9, zorder=6)

    best_line_ref, = ax.plot([], [], 'b-', linewidth=2, label='Best Path')
    curr_line_ref, = ax.plot([], [], 'r--', alpha=0.3, label='Searching')

    ax.legend(loc='lower right')
    ax.set_title("Initializing...")
    ax.axis('off')

    with plot_output:
        clear_output(wait=True)
        display(fig)

    plt.close(fig)

    population = create_population(pop_size)
    best_cost_overall = float('inf')
    best_individual_overall = ''

    for gen in range(generations):
        ranked_pop = calculate_cost_for_population(population, city_mapping, 'Delhi', dist_matrix)
        current_best = min(ranked_pop, key= lambda x: x[1])

        if (current_best[1] < best_cost_overall):
            best_cost_overall = current_best[1]
            best_individual_overall = current_best[0]

        next_gen = []
        while len(next_gen) < pop_size:
            p1 = tournament_selection(ranked_pop, tournament_size)
            p2 = tournament_selection(ranked_pop, tournament_size)
            c1, c2 = single_point_crossover(p1[0], p2[0], city_mapping, crossover_rate)
            next_gen.append(mutation(c1, mutation_rate))
            if len(next_gen) < pop_size: next_gen.append(mutation(c2, mutation_rate))

        population = next_gen

        if gen % 5 == 0 or gen == generations - 1:
            def get_xy(individual):
                chunks = [individual[i:i+4] for i in range(0, 56, 4)]
                names = ['Delhi'] + [city_mapping[c] for c in chunks] + ['Delhi']
                return [cities_coords[n][1] for n in names], [cities_coords[n][0] for n in names]

            x_curr, y_curr = get_xy(population[-1])
            x_best, y_best = get_xy(best_individual_overall)
            best_line_ref.set_data(x_best, y_best)
            curr_line_ref.set_data(x_curr, y_curr)
            ax.set_title(f"Gen {gen} | Best: {best_cost_overall:.0f} km")
            with plot_output:
                clear_output(wait=True)
                display(fig)

    chunks = [best_individual_overall[i:i+4] for i in range(0, 56, 4)]
    route_names = ['Delhi'] + [city_mapping[c] for c in chunks] + ['Delhi']
    with plot_output:
        print("\n" + "="*50)
        print(f"OPTIMIZED ROUTE FOUND")
        print("="*50)
        print(f"Total Distance: {best_cost_overall:.2f} km")
        print(f"Route: {' -> '.join(route_names)}")

### The Evolutionary Loop: Putting it All Together

The `run_ga` function is where the evolution actually happens. Here is the step-by-step logic of a single 'life cycle' in this notebook:

1.  **Map Initialization**: It first draws the map of India and plots the 15 cities as red dots to create the visual frame.
2.  **The First Generation**: It creates a random population of routes. Initially, these routes are very long and inefficient.
3.  **The Generation Loop**: For every generation requested (e.g., 50 times):
    *   **Evaluation**: It calculates the distance (cost) for every route in the current group.
    *   **Selection**: It uses the Tournament logic to pick the 'fittest' parents.
    *   **Crossover & Mutation**: It creates new 'child' routes by mixing parent genes and applying random swaps to maintain diversity.
4.  **Real-time Visualization**: Every 5 generations, the plot updates. The **blue line** shows the best route found so far, while the **faint red line** shows the algorithm actively 'searching' through other possibilities.
5.  **The Result**: Once the loop finishes, it prints out the final optimized path and the total distance in kilometers.

In [57]:
style = {'description_width':'initial'}
pop_slider = widgets.IntSlider(value=10, min=10, max=50, step=5, description='Population Size:', style=style)
gen_slider = widgets.IntSlider(value=10, min=5, max=50, step=5, description='Generations:', style=style)
cross_slider = widgets.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.05, description='Crossover Rate:', style=style)
mut_slider = widgets.FloatSlider(value=0.01, min=0.0, max=0.05, step=0.01, description='Mutation Rate:', style=style)
tourn_slider = widgets.IntSlider(value=3, min=2, max=6, step=1, description='Tournament Size:', style=style)


# Run Button
run_btn = widgets.Button(description="Run Algorithm", button_style='success')
def on_button_clicked(b):
    plot_output.clear_output()
    run_ga(
        pop_size=pop_slider.value,
        generations=gen_slider.value,
        crossover_rate=cross_slider.value,
        mutation_rate=mut_slider.value,
        tournament_size=tourn_slider.value,
    )

run_btn.on_click(on_button_clicked)

# Display Interface
display(
    widgets.VBox([
        pop_slider, gen_slider, cross_slider, mut_slider, tourn_slider,
        run_btn,
        plot_output
    ])
)

### Recommended Parameters for the 15-City TSP

While Genetic Algorithms involve randomness, these values typically perform well for this specific map:

*   **Population Size (30 - 40)**: A larger population explores more paths simultaneously. If it's too small (10), the algorithm might converge too quickly on a sub-optimal route.
*   **Generations (40 - 50)**: 15 cities create billions of possible route combinations. You need at least 40 generations to allow the 'good' genes to spread through the population.
*   **Crossover Rate (0.8 - 0.9)**: Most children should be a mix of their parents to keep the evolution moving forward.
*   **Mutation Rate (0.01 - 0.02)**: This should be **low**. High mutation turns the algorithm into a 'random search,' while 1-2% is just enough to prevent stagnation.
*   **Tournament Size (3)**: A size of 3 provides enough 'selection pressure' to favor the winners without completely ignoring the 'average' routes that might contain one or two great segments.

### Final Results and Conclusion

Once the generations are complete, the algorithm provides two main outputs:

1.  **The Visual Path**: The blue line on the map represents the shortest route the algorithm found during its entire run. It should look like a logical circuit around India.
2.  **The Technical Summary**: Below the map, the code prints the total distance and the exact sequence of cities.

Notice that the route starts and ends at **Delhi**, fulfilling our requirement for a 'round trip.' By adjusting the sliders and re-running the code, you can see how a larger population or more generations often lead to a significantly shorter total distance.